In [20]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Successfully uninstalled protobuf-3.20.3
  Using cached protobuf-3.20.3-py2.py3-none-any.whl.metadata (720 bytes)
Using cached protobuf-3.20.3-py2.py3-none-any.whl (162 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
ray 2.52.1 requires click!=8.3.*,>=7.0, but you have click 8.3.1 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobu

In [21]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [22]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import re
import seaborn as sns
import tokenize
import torch
import transformers

from datasets import Dataset

from math import ceil

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    TrainingArguments,
    Trainer,
    logging
)

from tqdm.auto import tqdm

(OBS) Code block added to delete the SQL status of the checkpoint (it has 7 GBs, so it ocuppies much space on the output). Add this code block at the beginning of the script, right after the imports.

In [23]:
path = "/kaggle/working/state.db"

if os.path.exists(path):
    os.remove(path)
    print("state.db deleted")
else:
    print("state.db not found")


state.db not found


In [24]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = f1_score(labels, predictions, average="macro")
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="weighted")
    recall = recall_score(labels, predictions, average="weighted")

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    }

In [25]:
MODEL_NAME = "microsoft/unixcoder-base"
MAX_LENGTH = 192

In [26]:
base_path = "/kaggle/input/sem-eval-2026-task-13-subtask-b/Task_B"

training_path = "/kaggle/input/sample/training_sample_set.parquet"
validation_path = "/kaggle/input/sample/validation_sample_set.parquet"
test_sample_path = base_path + "/test_sample.parquet"
test_full_path = base_path + "/test.parquet"

training_df = pd.read_parquet(training_path)
validation_df = pd.read_parquet(validation_path)
test_sample_df = pd.read_parquet(test_sample_path)
test_full_df = pd.read_parquet(test_full_path)

test_df = pd.merge(test_sample_df, test_full_df, on="code", how="inner")

In [27]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

The functions below preprocess code samples and erase the comments.

In [28]:
def is_escaped(result, i):
    count = 0
    i -= 1
    while i >= 0 and result[i] == '\\':
        count += 1
        i -= 1
    return count % 2 == 1

In [29]:
def strip_jcg_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    in_single_line_comment = False
    in_multi_line_comment = False
    string_delimiter = None
    in_verbatim_string = False

    while i < n:
        c = code[i]
        next_c = code[i + 1] if i + 1 < n else ''

        if in_single_line_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_single_line_comment = False
            i += 1
            continue

        if in_multi_line_comment:
            if c == '*' and next_c == '/':
                result[i] = result[i + 1] = ' '
                in_multi_line_comment = False
                i += 2
            else:
                if c != '\n':
                    result[i] = ' '
                i += 1
            continue

        if string_delimiter is not None:
            if string_delimiter == '`':
                if c == '`':
                    string_delimiter = None
            elif string_delimiter == '"':
                if c == '"' and not is_escaped(code, i):
                    string_delimiter = None
            elif string_delimiter == "'":
                if c == "'" and not is_escaped(code, i):
                    string_delimiter = None
            i += 1
            continue

        if in_verbatim_string:
            if c == '"' and next_c == '"':
                i += 2
            else:
                if c == '"' and next_c != '"':
                    in_verbatim_string = False
                i += 1
            continue

        if c == '@' and next_c == '"':
            in_verbatim_string = True
            i += 2
            continue

        if c == "'" or c == '"' or c == '`':
            string_delimiter = c
            i += 1
            continue

        if c == '/' and next_c == '/':
            result[i] = result[i + 1] = ' '
            in_single_line_comment = True
            i += 2
            continue

        if c == '/' and next_c == '*':
            result[i] = result[i + 1] = ' '
            in_multi_line_comment = True
            i += 2
            continue

        i += 1

    return ''.join(result)

In [30]:
def strip_php_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    in_single_line_comment = False
    in_multi_line_comment = False
    string_delimiter = None
    in_heredoc = False
    heredoc_id = None

    while i < n:
        c = code[i]
        next_c = code[i + 1] if i + 1 < n else ''

        if i == 0 or code[i - 1] == '\n':
            line_start = i
        else:
            line_start = None

        if in_heredoc:
            if line_start is not None:
                j = line_start
                k = 0
                while j < n and k < len(heredoc_id) and code[j] == heredoc_id[k]:
                    j += 1
                    k += 1
                if k == len(heredoc_id) and (j == n or code[j] in (';', '\n')):
                    in_heredoc = False
                    heredoc_id = None
            i += 1
            continue

        if in_single_line_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_single_line_comment = False
            i += 1
            continue

        if in_multi_line_comment:
            if c == '*' and next_c == '/':
                result[i] = result[i + 1] = ' '
                in_multi_line_comment = False
                i += 2
            else:
                if c != '\n':
                    result[i] = ' '
                i += 1
            continue

        if string_delimiter is not None:
            if c == string_delimiter and not is_escaped(code, i):
                string_delimiter = None
            i += 1
            continue

        if c == '<' and code[i:i+3] == '<<<':
            j = i + 3

            while j < n and code[j].isspace():
                j += 1

            if j < n and code[j] in ("'", '"'):
                quote = code[j]
                j += 1
                start = j
                while j < n and code[j] != quote:
                    j += 1
                heredoc_id = code[start:j]
                j += 1
            else:
                start = j
                while j < n and (code[j].isalnum() or code[j] == '_'):
                    j += 1
                heredoc_id = code[start:j]

            in_heredoc = True
            i = j
            continue


        if c == "'" or c == '"':
            string_delimiter = c
            i += 1
            continue

        if c == '/' and next_c == '/':
            result[i] = result[i + 1] = ' '
            in_single_line_comment = True
            i += 2
            continue

        if c == '#':
            result[i] = ' '
            in_single_line_comment = True
            i += 1
            continue

        if c == '/' and next_c == '*':
            result[i] = result[i + 1] = ' '
            in_multi_line_comment = True
            i += 2
            continue

        i += 1

    return ''.join(result)

In [31]:
def strip_python_comments(code: str) -> str:
    result = list(code)
    i = 0
    n = len(code)

    string_delimiter = None
    in_comment = False

    while i < n:
        c = code[i]

        if in_comment:
            if c != '\n':
                result[i] = ' '
            else:
                in_comment = False
            i += 1
            continue

        if string_delimiter is not None:
            if c == string_delimiter and not is_escaped(code, i):
                string_delimiter = None
            i += 1
            continue

        if c in ("'", '"'):
            string_delimiter = c
            i += 1
            continue

        if c == '#':
            result[i] = ' '
            in_comment = True
            i += 1
            continue

        i += 1

    return ''.join(result)

In [32]:
fallback_count = 0

def remove_python_docstrings(code: str) -> str:
    global fallback_count
    try:
        tokens = tokenize.generate_tokens(io.StringIO(code).readline)
        result = []
    
        scope_stack = [True]
    
        for tok in tokens:
            tok_type, tok_str, _, _, _ = tok
    
            if tok_type == tokenize.INDENT:
                scope_stack.append(True)
            elif tok_type == tokenize.DEDENT:
                scope_stack.pop()
            elif tok_type == tokenize.STRING and scope_stack[-1]:
                scope_stack[-1] = False
                continue
            elif tok_type not in (tokenize.NL, tokenize.NEWLINE):
                scope_stack[-1] = False
    
            result.append(tok)
    
        return tokenize.untokenize(result)

    except (IndentationError, SyntaxError, tokenize.TokenError):
        fallback_count += 1
        return code

In [33]:
def strip_python_comments_and_docstrings(code: str) -> str:
    code = strip_python_comments(code)
    code = remove_python_docstrings(code)
    return code

In [34]:
def normalize_whitespace(code: str) -> str:
    lines = code.splitlines()
    normalized = []

    for line in lines:
        stripped = line.rstrip()

        if stripped:
            m = re.match(r'^(\s*)(.*)$', stripped)
            indent, content = m.groups()
            content = re.sub(r' {2,}', ' ', content)
            normalized.append(indent + content)

    return '\n'.join(normalized)

In [35]:
def remove_jcg_comments(code: str) -> str:
    return normalize_whitespace(strip_jcg_comments(code))


def remove_php_comments(code: str) -> str:
    return normalize_whitespace(strip_php_comments(code))


def remove_python_comments(code: str) -> str:
    return normalize_whitespace(strip_python_comments_and_docstrings(code))

In [36]:
def clean_code(code: str, language: str) -> str:
    lang = language.lower() if isinstance(language, str) else ""
    if lang == "python":
        return remove_python_comments(code)
    elif lang == "php":
        return remove_php_comments(code)
    else:
        return remove_jcg_comments(code)

In [37]:
def preprocess_function(examples: pd.DataFrame):
    cleaned_code = [
        clean_code(code, lang)
        for code, lang in zip(examples["code"], examples["language"])
    ]

    examples["code"] = cleaned_code

    tokenized = tokenizer(
        examples["code"],
        truncation=True,
        max_length=MAX_LENGTH
    )

    return tokenized

In [38]:
def binary_label(generator):
    return 0 if generator == "Human" else 1

for df in (training_df, validation_df, test_df):
    df["binary_label"] = df["generator"].apply(binary_label)

(OBS) It is good practice to set the format to torch after you tokenize the datasets. Add the .set_format("torch") instructions right after you tokenized the datasets.

In [39]:
training_dataset = Dataset.from_pandas(training_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)

training_tokenized_set = training_dataset.map(preprocess_function, batched=True)
validation_tokenized_set = validation_dataset.map(preprocess_function, batched=True)
test_tokenized_set = test_dataset.map(preprocess_function, batched=True)

training_tokenized_set.set_format("torch")
validation_tokenized_set.set_format("torch")
test_tokenized_set.set_format("torch")

AttributeError: 'Series' object has no attribute 'columns'

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("\nRunning on device:", device)
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

In [ ]:
print("Docstring fallback count:", fallback_count)

In [ ]:
logging.set_verbosity_info()

In [ ]:
class TrainingProgressCallback(transformers.TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.is_local_process_zero:
            tqdm.write(f"Step {state.global_step}/{state.max_steps}")

In [ ]:
binary_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

binary_model.to(device)
binary_model.gradient_ckeckpointing_enable()

In [ ]:
binary_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints_A",
    seed=42,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_dir="/kaggle/working/logs",
    logging_steps=3000,
    fp16=True,
    disable_tqdm=False,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    no_cuda=False,
    report_to=[]
)

In [ ]:
binary_trainer = Trainer(
    model=binary_model,
    args=binary_args,
    train_dataset=training_tokenized_set,
    eval_dataset=validation_tokenized_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
                   early_stopping_patience=1,
                   early_stopping_threshold=0.001
               ),
               TrainingProgressCallback()]
)

In [ ]:
binary_trainer.train()

In [ ]:
with open("taskA_training_results.pkl", "wb") as f:
    pickle.dump(binary_trainer.state.log_history, f)

In [ ]:
llm_training_df = training_df[train_df.generator != "Human"].copy()
llm_validation_df = validation_df[val_df.generator != "Human"].copy()
llm_test_df = test_df[test_df.generator != "Human"].copy()

In [ ]:
llm_training_dataset = Dataset.from_pandas(llm_training_df)
llm_validation_dataset = Dataset.from_pandas(llm_validation_df)
llm_test_dataset = Dataset.from_pandas(llm_test_df)

llm_training_tokenized_set = llm_training_dataset.map(preprocess_function, batched=True)
llm_validation_tokenized_set = llm_validation_dataset.map(preprocess_function, batched=True)
llm_test_tokenized_set = llm_test_dataset.map(preprocess_function, batched=True)

llm_training_tokenized_set.set_format("torch")
llm_validation_tokenized_set.set_format("torch")
llm_test_tokenized_set.set_format("torch")

In [ ]:
id2label = {
    1: "deepseek-ai/DeepSeek-V3-0324",
    2: "Qwen/Qwen2.5-Coder-7B-Instruct",
    3: "01-ai/Yi-Coder-9B-Chat",
    4: "bigcode/starcoder",
    5: "gemma-3n-e4b-it",
    6: "microsoft/phi-2",
    7: "meta-llama/Llama-3.3-70B-Instruct-Turbo",
    8: "ibm-granite/granite-3.2-2b-instruct",
    9: "mistralai/Devstral-Small-2505",
    10: "GPT-4o-mini"
}

label2id = {
    "deepseek-ai/DeepSeek-V3-0324": 1,
    "Qwen/Qwen2.5-Coder-7B-Instruct": 2,
    "01-ai/Yi-Coder-9B-Chat": 3,
    "bigcode/starcoder": 4,
    "gemma-3n-e4b-it": 5,
    "microsoft/phi-2": 6,
    "meta-llama/Llama-3.3-70B-Instruct-Turbo": 7,
    "ibm-granite/granite-3.2-2b-instruct": 8,
    "mistralai/Devstral-Small-2505": 9,
    "GPT-4o-mini": 10
}

In [ ]:
llm_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=10,
    id2label=id2label,
    label2id=label2id
)

llm_model.to(device)
llm_model.gradient_checkpointing_enable()

(OBS) In the TrainingArguments parameters, you have to set the following:
- output_dir="/kaggle/working/checkpoints" (this is the Kaggle notebook folder that stores the output files; you will need to have persistent data, to achieve that the checkpoints must be saved in this folder);
- num_train_epochs=3;
- load_best_model_at_end=True (for best results);
- eval_strategy="epoch";
- save_strategy="epoch";
- save_total_limit=2 (or 3, you set here the last number of checkpoints that remain saved);
- logging_dir="/kaggle/working/logs" (optional, stores log information);
- logging_steps=3000 (optional, add only if you put logging_dir too);

In [ ]:
llm_training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints_B",
    seed=42,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_dir="/kaggle/working/logs",
    logging_steps=3000,
    fp16=True,
    disable_tqdm=False,
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    no_cuda=False,
    report_to=[]
)

In [ ]:
counts = torch.tensor(
    llm_training_df["labels"].value_counts().sort_index().values,
    dtype=torch.float
)

class_weights = (1.0 / counts).sqrt()
class_weights = class_weights / class_weights.sum()

In [ ]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_function = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_function(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
llm_trainer = WeightedTrainer(
    model=llm_model,
    args=llm_training_args,
    train_dataset=llm_training_tokenized_set,
    eval_dataset=llm_validation_tokenized_set,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
                   early_stopping_patience=1,
                   early_stopping_threshold=0.001
               ),
               TrainingProgressCallback()]
)

(OBS) The first execution will have just trainer.train(), because initially you won't have any checkpoints, you start from 0 with the training. After that, you need to replace this instruction with trainer.train(resume_from_checkpoint=True), such that it resumes the training from the last checkpoint.

In [ ]:
llm_trainer.train()

In [ ]:
with open("taskB_training_results.pkl", "wb") as f:
    pickle.dump(llm_trainer.state.log_history, f)

In [ ]:
binary_evaluation = binary_trainer.evaluate(test_tokenized_set)
print("Evaluation for task A: ", binary_evaluation)

binary_preds = np.argmax(
    binary_trainer.predict(test_tokenized_set).predictions, axis=1
)

llm_evaluation = llm_trainer.evaluate(llm_test_tokenized_set)
print("Evaluation for task B: ", llm_evaluation)

llm_preds = np.argmax(
    llm_trainer.predict(llm_test_tokenized_set).predictions, axis=1
)

final_preds = []
llm_index = 0
for p in binary_preds:
    if p == 0:
        final_preds.append("Human")
    else:
        final_preds.append(id2label[llm_preds[llm_index]])
        llm_idx += 1


In [ ]:
pd.DataFrame({
    "ID": test_df["ID"],
    "label": binary_preds
}).to_csv("taskA_predictions.csv", index=False)

pd.DataFrame({
    "ID": test_df["ID"],
    "label": final_preds
}).to_csv("taskB_predictions.csv", index=False)